In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.spatial.distance import euclidean
import numpy as np

csv_file = '/content/drive/MyDrive/converted_colors.csv'
df = pd.read_csv(csv_file, encoding='utf-8-sig')

In [4]:
print(df.head())

                       File Name  color  Color Index  Red  Green  Blue  \
0    3CE_드롭글로우젤_CALMING    NaN            1  163     79    96   
1  3CE_드롭글로우젤_ESSENTIAL    NaN            1  166     84    88   
2     3CE_드롭글로우젤_MILDER    NaN            1  185     81   110   
3       3CE_드롭글로우젤_NEAT    NaN            1  195     77    92   
4   3CE_드롭글로우젤_RAWAPPLE    NaN            1  209     36    73   

  Max Proportion      HEX   Price formulation  date  saturation  lightness  \
0        100.00%  #a34f60  18,000         글로시     5          52         47   
1         58.71%  #a65458  18,000         글로시     5          49         49   
2        100.00%  #b9516e  18,000         글로시     5          56         52   
3         93.65%  #c34d5c  18,000         글로시     5          61         53   
4         70.15%  #d12449  18,000         글로시     5          83         48   

     tone  
0  strong  
1    dull  
2  strong  
3  strong  
4   vivid  


In [ ]:
#tone

# def find_tone(FileName, df):
#     product_row = df[df['File Name'].str.lower() == FileName.lower()]
#     if product_row.empty:
#         raise ValueError(f"Product '{FileName}' not found in the dataset.")
#     tone = product_row.iloc[0]['tone']
#     return tone

In [ ]:
# input_product = input("Enter the product name: ")
# try:
#     product_tone = find_tone(input_product, df)
#     print(f"The tone of '{input_product}' is '{product_tone}'.")
# except ValueError as e:
#     print(e)

In [5]:
import unicodedata

df['Normalized File Name'] = df['File Name'].apply(lambda x: unicodedata.normalize('NFC', x))

In [6]:
# RGB 값 추출 및 유클리드 거리 계산
def calculate_rgb_similarity(input_product, df):
    input_row = df[df['Normalized File Name'].str.lower() == input_product.lower()]
    if input_row.empty:
        raise ValueError(f"Product '{input_product}' not found.")

    input_rgb = input_row[['Red', 'Green', 'Blue']].values.flatten()
    df['Distance'] = df[['Red', 'Green', 'Blue']].apply(lambda x: euclidean(input_rgb, x), axis=1)
    return df

# Formulation과 Tone의 유사도 계산
def calculate_formulation_tone_similarity(input_product, df):
    input_row = df[df['Normalized File Name'].str.lower() == input_product.lower()]
    if input_row.empty:
        raise ValueError(f"Product '{input_product}' not found.")

    input_formulation = input_row.iloc[0]['formulation']
    input_tone = input_row.iloc[0]['tone']
    df['formulation Similarity'] = df['formulation'].apply(lambda x: 1 if x == input_formulation else 0)
    df['tone Similarity'] = df['tone'].apply(lambda x: 1 if x == input_tone else 0)
    return df

# 종합 유사도 계산
def total_similarity(df):
    df['Total Similarity'] = (
        (1 - df['Distance'] / np.max(df['Distance'])) * 0.4 +
        df['formulation Similarity'] * 0.2 +
        df['tone Similarity'] * 0.4
    )
    return df.sort_values(by='Total Similarity', ascending=False)  #내림차순 정렬

# 제품 추천 함수
def recommend_products(input_product, df, num_recommendations=5):
    try:
        df = calculate_rgb_similarity(input_product, df)
        df = calculate_formulation_tone_similarity(input_product, df)
        df = total_similarity(df)

        recommendations = df[df['Normalized File Name'].str.lower() != input_product.lower()]
        return recommendations[['Normalized File Name', 'Total Similarity']].head(num_recommendations)
    except ValueError as e:
        return str(e)

# 사용자 입력 받기
input_product = input("Enter the product name:")
#input_product = input_product.encode('cp949').encode('utf-8')

print(f"입력된 키워드: {repr(input_product)}")


#with open('euc_kr.txt', encoding='euc-kr', mode='w') as f:
#    f.write(input_product)

def search_data(df, input_product):
    results = df[df['File Name'].str.contains(input_product, na=False)]
    return results


Enter the product name:데이지크_크림드버터틴트_코튼플럼
입력된 키워드: '데이지크_크림드버터틴트_코튼플럼'


In [7]:
# 제품 추천 실행
recommendations = recommend_products(input_product, df)
print("추천 제품:")
print(recommendations)

추천 제품:
        Normalized File Name  Total Similarity
605      무지개맨션_무드웨어블러립스틱_리벤지          0.994008
1497         힌스_슬림핏리퀴드벨벳_블루미          0.987466
665          뮤드_소프트블러틴트_페일페탈          0.986107
686   바닐라코_벨벳블러드베일립스틱_위스퍼링핑크          0.983727
1337          플린_브리즈벨벳틴트_딤로즈          0.982274


In [8]:
def get_price_by_normalized_file_name(file_name):
    # 입력된 Normalized File Name과 일치하는 행을 찾음
    result = df[df['Normalized File Name'] == file_name]

    # 해당하는 값이 존재하는지 확인
    if not result.empty:
        # Price 값을 출력
        price = result['Price'].values[0]
        print(f"Price: {price}")
    else:
        print(f"No matching Normalized File Name found for '{file_name}'.")

# 예시: 사용자가 입력한 Normalized File Name
user_input = input("Enter the Normalized File Name: ")

# 해당 Normalized File Name에 해당하는 Price를 출력
get_price_by_normalized_file_name(user_input)

Enter the Normalized File Name: 데이지크_크림드버터틴트_코튼플럼
Price: 17,000


In [9]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os


#for index, row in df.iterrows():
  #product_name = os.path.splitext(row['Normalized File Name'])[0]
  #product_price = df.loc[df['Normalized File Name'] == row['Normalized File Name'], 'Price'].values[0]
  #price = df['Price']

  #merged_df = pd.merge(df, df[['Normalized File Name', 'Price']], left_on='Normalized File Name', right_on='Normalized File Name', how='left')
  #price = merged_df['Price_x']
  #formulation = row['formulation']



# 제품 추천 함수
def recommend_products(input_product, df, num_recommendations=10):
    try:
        df = calculate_rgb_similarity(input_product, df)
        df = calculate_formulation_tone_similarity(input_product, df)
        df = total_similarity(df)

        recommendations = df[df['Normalized File Name'].str.lower() != input_product.lower()]
        recommended_products = recommendations[['Normalized File Name', 'Total Similarity']].head(num_recommendations)

        for index, row in recommended_products.iterrows():
            product_name = os.path.splitext(row['Normalized File Name'])[0]
            normalized_file_name = row['Normalized File Name']

            price_row = df[df['Normalized File Name'] == normalized_file_name]['Price']

            if not price_row.empty:
                price = price_row.iloc[0]
            else:
                price = "Price not found"

            formulation_row = df[df['Normalized File Name'] == normalized_file_name]['formulation']
            if not formulation_row.empty:
                formulation = formulation_row.iloc[0]
            else:
                formulation = "Formulation not found"

            tone_row = df[df['Normalized File Name'] == normalized_file_name]['tone']
            if not tone_row.empty:
                tone = tone_row.iloc[0]
            else:
                tone = "Tone not found"

            print()
            print("Name:", product_name)
            print("Price:", price)
            print("Formulation:", formulation)
            print("Tone:", tone)
            print()
            image_path = f"/content/drive/MyDrive/oliveyoung_lip_color/{row['Normalized File Name']}"
            img = mpimg.imread(image_path)
            plt.imshow(img)
            plt.axis('off')
            plt.show()

    except ValueError as e:
        print(e)

# 사용자 입력
input_product = input("제품명을 입력하세요: ")

# 제품 추천
recommend_products(input_product, df)

제품명을 입력하세요: 데이지크_크림드버터틴트_코튼플럼

Name: 무지개맨션_무드웨어블러립스틱_리벤지
Price: 23,000
Formulation: 매트
Tone: strong



FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/oliveyoung_lip_color/무지개맨션_무드웨어블러립스틱_리벤지'